# CS3081 — Assignment 3: Uncertainty and Bayesian Networks
**Hashim AlSayed · S24109290 · Effat University**

## Exercise 5(c) — Exact inference and rejection sampling
The network is H → A, H → C, A → C, C → L. All states use the explicit order `yes`, `no`; all reported probabilities have four decimal places. Calculations retain full precision internally.

Run all cells in VS Code or execute with `jupyter nbconvert --to notebook --execute --inplace assignment_bn.ipynb`. Dependencies: `pgmpy`, `numpy`, `pandas`, `ipykernel`; package versions are printed below.

In [1]:
import sys
import importlib.metadata as metadata
from itertools import product
from fractions import Fraction as F
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
print("Python:", sys.version.split()[0])
for package in ["pgmpy", "numpy", "pandas"]:
    print(f"{package}: {metadata.version(package)}")

Python: 3.13.5
pgmpy: 1.1.2
numpy: 2.5.4
pandas: 3.1.1


/workspace/ai_a3/.venv/lib/python3.13/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


### 1. Build and validate the model
For powercut the parent order is `[heatwave, overload]`: its columns are `(yes,yes)`, `(yes,no)`, `(no,yes)`, `(no,no)`. Each second row is the complement of its first row.

In [2]:
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"), ("heatwave", "powercut"),
    ("overload", "powercut"), ("powercut", "cancel")
])
states = {node: ["yes", "no"] for node in model.nodes()}
cpd_h = TabularCPD("heatwave", 2, [[0.3], [0.7]],
                   state_names={"heatwave": states["heatwave"]})
cpd_a = TabularCPD("overload", 2, [[0.6, 0.1], [0.4, 0.9]],
    evidence=["heatwave"], evidence_card=[2],
    state_names={k: states[k] for k in ["overload", "heatwave"]})
cpd_c = TabularCPD("powercut", 2,
    [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
    evidence=["heatwave", "overload"], evidence_card=[2, 2],
    state_names={k: states[k] for k in ["powercut", "heatwave", "overload"]})
cpd_l = TabularCPD("cancel", 2, [[0.8, 0.1], [0.2, 0.9]],
    evidence=["powercut"], evidence_card=[2],
    state_names={k: states[k] for k in ["cancel", "powercut"]})
model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print(model.check_model())
for cpd in model.get_cpds():
    print(cpd)

True
+---------------+-----+
| heatwave(yes) | 0.3 |
+---------------+-----+
| heatwave(no)  | 0.7 |
+---------------+-----+
+---------------+---------------+--------------+
| heatwave      | heatwave(yes) | heatwave(no) |
+---------------+---------------+--------------+
| overload(yes) | 0.6           | 0.1          |
+---------------+---------------+--------------+
| overload(no)  | 0.4           | 0.9          |
+---------------+---------------+--------------+
+---------------+---------------+---------------+---------------+--------------+
| heatwave      | heatwave(yes) | heatwave(yes) | heatwave(no)  | heatwave(no) |
+---------------+---------------+---------------+---------------+--------------+
| overload      | overload(yes) | overload(no)  | overload(yes) | overload(no) |
+---------------+---------------+---------------+---------------+--------------+
| powercut(yes) | 0.7           | 0.2           | 0.5           | 0.05         |
+---------------+---------------+-------------

### 2. Exact inference with VariableElimination
The hand result is `(27/200) / (5031/20000) = 900/1677`, reported as **0.5367**.

In [3]:
inference = VariableElimination(model)
posterior = inference.query(variables=["heatwave"], evidence={"cancel": "yes"}, show_progress=False)
print(posterior)
exact = float(posterior.get_value(heatwave="yes"))
print(f"P(H=yes | L=yes) = {exact:.4f}")
assert np.isclose(exact, float(F(900, 1677)), rtol=0, atol=1e-12)

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
P(H=yes | L=yes) = 0.5367


### 3. Rejection sampling: 10,000 retained samples
The assignment hint specifies `size=10000`: pgmpy returns 10,000 samples **consistent with cancel=yes**, not 10,000 prior proposals before rejection. Seed 42 makes this run reproducible within the recorded environment; every returned row is checked against the evidence.

In [4]:
samples = BayesianModelSampling(model).rejection_sample(
    evidence=[State("cancel", "yes")], size=10000,
    seed=42, show_progress=False
)
assert len(samples) == 10000
assert samples["cancel"].eq("yes").all()
yes_count = int(samples["heatwave"].eq("yes").sum())
estimate = yes_count / len(samples)
print(samples.head(10).to_string(index=False))
print(f"Retained samples: {len(samples)}")
print(f"Heatwave=yes count: {yes_count}")
print(f"Sampling estimate: {estimate:.4f}")
print(f"Exact probability: {exact:.4f}")
print(f"Absolute error: {abs(estimate-exact):.4f}")
print(f"Estimated standard error: {np.sqrt(estimate*(1-estimate)/len(samples)):.4f}")

heatwave overload powercut cancel
     yes      yes      yes    yes
     yes      yes      yes    yes
     yes       no      yes    yes
      no       no       no    yes
     yes      yes      yes    yes
      no       no       no    yes
     yes      yes      yes    yes
     yes      yes      yes    yes
      no       no      yes    yes
     yes       no      yes    yes
Retained samples: 10000
Heatwave=yes count: 5430
Sampling estimate: 0.5430
Exact probability: 0.5367
Absolute error: 0.0063
Estimated standard error: 0.0050


### 4. Two-sentence comparison
The exact probability is 0.5367, whereas the rejection-sampling estimate is 0.5430, an absolute difference of 0.0063. This difference is due to finite random sampling; increasing the number of retained samples generally reduces sampling error and makes the estimate approach the exact value.

## Numerical verification of the written answers
The following optional verification cells independently enumerate all 16 complete assignments using exact rational arithmetic, compare all overlapping network answers with pgmpy, and verify Exercises 1, 2 and the supplied small sample tables. The eight displayed rows are the unnormalized terms with L=yes; four-place displays do not truncate the underlying fractions.

In [5]:
def joint(h, a, c, l):
    ph = F(3,10) if h == "yes" else F(7,10)
    ay = F(6,10) if h == "yes" else F(1,10)
    pa = ay if a == "yes" else 1-ay
    cy = {("yes","yes"):F(7,10), ("yes","no"):F(2,10),
          ("no","yes"):F(5,10), ("no","no"):F(5,100)}[h,a]
    pc = cy if c == "yes" else 1-cy
    ly = F(8,10) if c == "yes" else F(1,10)
    pl = ly if l == "yes" else 1-ly
    return ph*pa*pc*pl
joint_table = {s: joint(*s) for s in product(["yes", "no"], repeat=4)}
assert sum(joint_table.values()) == 1

def mass(**evidence):
    indexes = {"heatwave":0, "overload":1, "powercut":2, "cancel":3}
    return sum(p for s,p in joint_table.items()
               if all(s[indexes[k]] == v for k,v in evidence.items()))

def conditional(query, evidence):
    return mass(**{**evidence, **query}) / mass(**evidence)

rows = []
for (h,a,c,l), weight in joint_table.items():
    if l == "yes":
        rows.append([h,a,c,str(weight),f"{float(weight):.4f}"])
print(pd.DataFrame(rows, columns=["H","A","C","Exact weight","4 d.p."]).to_string(index=False))
print("P(L=yes), exact fraction:", mass(cancel="yes"))
checks = [
    ("E3(4)", mass(heatwave="yes", overload="no", powercut="yes", cancel="yes"), F(192,10000)),
    ("E3(5), H|C", conditional({"heatwave":"yes"},{"powercut":"yes"}), F(300,433)),
    ("E3(5), H|C,L", conditional({"heatwave":"yes"},{"powercut":"yes","cancel":"yes"}), F(300,433)),
    ("E4(1), H|L", conditional({"heatwave":"yes"},{"cancel":"yes"}), F(900,1677)),
    ("E4(2), A|L", conditional({"overload":"yes"},{"cancel":"yes"}), F(918,1677)),
    ("E4(3), H|L,A=no", conditional({"heatwave":"yes"},{"cancel":"yes","overload":"no"}), F(192,759)),
]
for label, actual, expected in checks:
    assert actual == expected, (label, actual, expected)
    print(f"{label}: {float(actual):.4f} — exact fraction {actual}; PASS")
for variable, evidence, expected in [
    ("heatwave", {"powercut":"yes"}, F(300,433)),
    ("heatwave", {"powercut":"yes", "cancel":"yes"}, F(300,433)),
    ("heatwave", {"cancel":"yes"}, F(900,1677)),
    ("overload", {"cancel":"yes"}, F(918,1677)),
    ("heatwave", {"cancel":"yes", "overload":"no"}, F(192,759))
]:
    result = inference.query([variable], evidence=evidence, show_progress=False)
    value = float(result.get_value(**{variable:"yes"}))
    assert np.isclose(value, float(expected), rtol=0, atol=1e-12)
print("All overlapping hand/enumeration/pgmpy checks PASS.")

  H   A   C Exact weight 4 d.p.
yes yes yes       63/625 0.1008
yes yes  no      27/5000 0.0054
yes  no yes       12/625 0.0192
yes  no  no        6/625 0.0096
 no yes yes        7/250 0.0280
 no yes  no       7/2000 0.0035
 no  no yes      63/2500 0.0252
 no  no  no   1197/20000 0.0599
P(L=yes), exact fraction: 5031/20000
E3(4): 0.0192 — exact fraction 12/625; PASS
E3(5), H|C: 0.6928 — exact fraction 300/433; PASS
E3(5), H|C,L: 0.6928 — exact fraction 300/433; PASS
E4(1), H|L: 0.5367 — exact fraction 300/559; PASS
E4(2), A|L: 0.5474 — exact fraction 306/559; PASS
E4(3), H|L,A=no: 0.2530 — exact fraction 64/253; PASS
All overlapping hand/enumeration/pgmpy checks PASS.


In [6]:
e1 = {
    "Joint total": F(56,100)+F(14,100)+F(12,100)+F(18,100),
    "P(dusty)": F(30,100), "P(late)": F(32,100),
    "P(late|dusty)": F(18,30), "P(late|clear)": F(14,70),
    "P(dusty|late)": F(18,32)
}
free = F(6,10)*F(3,10)+F(5,100)*F(7,10)
both_spam = F(3,10)*F(6,10)*F(4,10)
both_normal = F(7,10)*F(5,100)*F(2,100)
e2 = {"P(free)":free, "P(spam|free)":F(18,100)/free,
      "P(spam|free,winner)":both_spam/(both_spam+both_normal)}
for label,value in {**e1, **e2}.items():
    print(f"{label} = {float(value):.4f}")
assert e2["P(spam|free)"] == F(36,43)
assert e2["P(spam|free,winner)"] == F(720,727)
# Transcribe the supplied tables rather than generate new samples.
rejection = pd.DataFrame([
["no","no","no","no"],["yes","yes","yes","yes"],
["no","no","no","no"],["no","yes","yes","yes"],
["yes","yes","no","no"],["no","no","no","yes"],
["yes","no","yes","yes"],["no","no","no","no"],
["no","no","no","no"],["yes","yes","yes","no"],
["no","no","no","no"],["no","no","yes","yes"]],
columns=["H","A","C","L"], index=range(1,13))
keep = rejection[rejection.L == "yes"]
print("Keep:", keep.index.tolist())
print("Reject:", rejection[rejection.L != "yes"].index.tolist())
assert keep.index.tolist() == [2,4,6,7,12]
assert F(int(keep.H.eq("yes").sum()), len(keep)) == F(2,5)
weights = [F(1,10), F(8,10), F(1,10), F(1,10), F(8,10), F(8,10)]
weighted_yes = sum(weights[i] for i in [1,3,5])
weighted_estimate = weighted_yes / sum(weights)
assert weighted_estimate == F(17,27)
print(f"E5(a) estimate = {2/5:.4f}; rejection fraction = {7/12:.4f}")
print("E5(b) weights:", ", ".join(f"{float(w):.4f}" for w in weights))
print(f"E5(b) estimate = {float(weighted_estimate):.4f}")
print("All rational-arithmetic checks PASS.")

Joint total = 1.0000
P(dusty) = 0.3000
P(late) = 0.3200
P(late|dusty) = 0.6000
P(late|clear) = 0.2000
P(dusty|late) = 0.5625
P(free) = 0.2150
P(spam|free) = 0.8372
P(spam|free,winner) = 0.9904
Keep: [2, 4, 6, 7, 12]
Reject: [1, 3, 5, 8, 9, 10, 11]
E5(a) estimate = 0.4000; rejection fraction = 0.5833
E5(b) weights: 0.1000, 0.8000, 0.1000, 0.1000, 0.8000, 0.8000
E5(b) estimate = 0.6296
All rational-arithmetic checks PASS.


### Interpretation and references
The six-sample likelihood-weighting estimate is 0.6296 and the supplied rejection estimate is 0.4000; both differ from the exact 0.5367 because the samples are small. The `size=10000` interpretation follows the assignment's explicit pgmpy call and its returned table; the actual number of rejected prior proposals is not reported by that call.

Sources: provided eight-page CS3081 assignment; [pgmpy sampling documentation](https://pgmpy.org/api/generated/inference/pgmpy.sampling.BayesianModelSampling.html) and [TabularCPD documentation](https://pgmpy.org/api/generated/factors/pgmpy.factors.discrete.TabularCPD.html).